# Build a Tool-Calling Agent

**Course author:** Jayneel Girnara

Introduce tools as ordinary Python functions with schemas, then give those tools to a LangChain agent so it can decide when to call them.

## Learning goals
- Define typed tools and document their purpose.
- Test a tool directly before connecting it to an agent.
- Create an agent with a chat model and inspect its final response.

## Before you start
Install the dependencies and configure `OPENAI_API_KEY` in a local .env file. Do not commit credentials. Tool calls to the hosted model may incur charges.

## How to work through it
Run cells top to bottom: define and test the tools, configure the model, create the agent, and invoke it with a question that requires arithmetic. Compare the agent result with the direct tool result.

## 1. Define and test tools

A **tool** is a Python function exposed to a model with a name, description, and input schema. The `@tool` decorator uses type annotations and the docstring to describe inputs and purpose. Testing the function directly verifies its basic behavior without involving an LLM.

**Checkpoint:** What information helps the model decide when and how to call a tool?

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_core.tools import tool

In [9]:
@tool
def addnumber(a:int,b:int):
    """
    it will return sum of two numbers
    Args:
        a:Number One
        b:Number Two
    """
    return a+b
@tool
def multiply(a:int,b:int):
    """
    it will return multiplication of two numbers
    Args:
        a:Number One
        b:Number Two
    """
    return a*b

In [4]:
addnumber.invoke({"a":12,"b":3})

15

## 2. Create an agent with tools

An **agent** combines a model with an allowed set of tools. The model can request a tool call; the agent runtime executes that call and returns the result to the model. `system_prompt` describes the task and `tools` limits available capabilities. The agent is not guaranteed to call tools correctly, so keep tools narrow and results reviewable.

**Checkpoint:** How is an agent different from calling `multiply` directly?

In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

In [ ]:
llm=ChatOpenAI(model="gpt-4o-mini")

In [12]:
agent=create_agent(
    model=llm,
    tools=[addnumber,multiply],
    system_prompt="You are math teacher, and always use tools for calculations."
)

## 3. Ask a question that requires a tool

This arithmetic question is intended to make the agent use the provided functions. Inspect the final assistant message and, when debugging, the intermediate messages to see whether a tool was actually called. For important calculations, verify the result independently.

**Checkpoint:** Did the agent use the right tool for the calculation?

In [18]:
response=agent.invoke({"messages":[{"role":"user","content":"What is 2 + 30*10?"}]})

In [19]:
print(response['messages'][-1].content)

The result of 2 + 30*10 is 302.
